In [1]:
import pandas as pd
import os
import qa_functions as qa
from tqdm import tqdm
import temperature_data_fns as td

pd.set_option("display.max_columns", None)

# Define string to be removed/added in the cleaning process. Used for naming outputs:
prefix = "Property_ID="

In [2]:
# Set file format of cleaned data you have, either parquet or csv
# If you have run the clean.ipynb file to generate the cleaned data, they will be in parquet
file_format = qa.set_file_format(file_format="parquet")

In [3]:
# Specify which folder contains your local directory
location = os.environ["EoH"]

# Specify location of specific folders
location_out = os.path.join(location, "processed")
location_out_cleaned = os.path.join(location_out, "cleaned")

# Generate a list of all files to be run through the code
files_info = []

for home in os.listdir(os.path.join(location_out_cleaned, "cleaned")):
    size = os.path.getsize(os.path.join(location_out_cleaned, "cleaned", home))
    file_info = [home, round(size / 1024)]
    files_info.append(file_info)
# Remove properties which do not have more than 1KB of data
files_info = pd.DataFrame(files_info, columns=["Property_Name", "Size_KB"])
files_info = files_info[files_info["Size_KB"] > 1]
files_list = list(files_info.iloc[:, 0])
files_list = [file.split(".")[0] for file in files_list]

In [4]:
# Define the boundaries for the ranges of gap length
# duration < short - Not a gap
# short < duration < medium - a short gap
# medium < duration < long - a medium gap
# duration > long - a long gap
gap_len_defs = {
    "long": pd.Timedelta(days=21),
    "medium": pd.Timedelta(days=7),
    "short": pd.Timedelta(minutes=30),
}

In [5]:
# Define the performance factor ranges for different time scales
# For short time scales, we expect higher variation in the performance factor
# short time periods are around a day, long are around a year, medium is in between these two
spf_ranges = {
    "short": {"min": 0.75, "max": 7.5},
    "medium": {"min": 0.9, "max": 6.5},
    "long": {"min": 1.5, "max": 5.0},
}

In [6]:
window_method = "best"
save_scored_data = False
plotting = False
all_windows = pd.DataFrame()
home_summary = []
all_homes_alteration_record = pd.DataFrame()
home_summary_cleaned = pd.read_csv(os.path.join(location, "processed", "home_summary_partial_1.csv"))
for file_home in tqdm(files_list):
    if plotting:
        # Un-comment to save the plots
        plot_full_save_path = os.path.join(location_out_cleaned, "plots", "full", file_home + ".png")
        plot_window_save_path = os.path.join(location_out_cleaned, "plots", "window", file_home + ".png")
        # Un-comment for fig.show()
        # plot_full_save_path = "none"
        # plot_window_save_path = "none"
    else:
        plot_full_save_path = ""
        plot_window_save_path = ""

    if file_format == "csv":
        data = pd.read_csv(os.path.join(location_out_cleaned, "cleaned", f"{file_home}.csv")).set_index("Timestamp")
    elif file_format == "parquet":
        data = pd.read_parquet(os.path.join(location_out_cleaned, "cleaned", f"{file_home}.parquet")).set_index("Timestamp")

    home_id = file_home.replace(prefix, "")
    home_summary_part = home_summary_cleaned[home_summary_cleaned["Property_ID"] == home_id]

    # Select the best window for this data, this function also adds stats about the window to the output
    (window_data, home_summary_part, cleaned_data, windows, single_home_alteration_record) = qa.select_window(
        data,
        gap_len_defs,
        home_summary_part,
        spf_ranges=spf_ranges,
        window_len_mths=12,
        method=window_method,
        plot_full_save_path=plot_full_save_path,
        plot_window_save_path=plot_window_save_path,
        location_out_cleaned=location_out_cleaned,
        file=file_home,
        save_scored_data=save_scored_data,
    )

    # Add most common flow temperatures to home_summary
    file_path = os.path.join(location_out, "binned_heating_temperature", file_home + ".csv")
    plot_path = os.path.join(location_out, "binned_heating_temperature", "plots", file_home + ".png")

    # We only want to add these stats if a window exists
    if "window_start" in home_summary_part.columns:
        home_summary_part = td.add_flow_temp_stats_for_window(
            data, home_summary_part, home_id, file_path=file_path, plot_path=plot_path
        )

    # Find spfs for coldest day
    home_summary_part_cold_day = td.add_spfs_for_coldest_period(data, home_id, pd.Timedelta(days=1), "Coldest_day_")

    # Find spfs for coldest half-hour
    home_summary_part_cold_HH = td.add_spfs_for_coldest_period(data, home_id, pd.Timedelta(minutes=30), "Coldest_HH_")

    if (len(home_summary_part_cold_day) > 0) & (len(home_summary_part_cold_HH) > 0):
        home_summary_part_cold = pd.merge(
            home_summary_part_cold_day,
            home_summary_part_cold_HH,
            on="Property_ID",
            how="outer",
        )

        home_summary_part = home_summary_part.merge(home_summary_part_cold, on="Property_ID")

    home_summary_part["window_method"] = window_method
    home_summary.append(home_summary_part)

    # We want to save all the possible windows out separately to do some analysis on them
    windows["Property_ID"] = home_id
    all_windows = pd.concat([all_windows, windows], axis=0)

    # We want to keep all the alteration records for all homes
    if len(single_home_alteration_record) > 0:
        single_home_alteration_record["Property_ID"] = home_id
    all_homes_alteration_record = pd.concat([all_homes_alteration_record, single_home_alteration_record])

home_summary = pd.concat(home_summary)

cleaning_flags = pd.read_csv(os.path.join(location_out, "temperature_stats_with_outcome.csv"))
cleaning_flags = (
    cleaning_flags.groupby("Property_ID")
    .max()[["issue", "outcome", "anomalies cleaned"]]    
    .add_prefix("temperature_cleaning_")
    .reset_index()
)

home_summary = pd.merge(home_summary, cleaning_flags, on="Property_ID")

home_summary

  0%|          | 0/739 [00:00<?, ?it/s]c:\Users\irene.garcia\Documents\DS_analysis_tests\review\notebooks\data_scoring_fns.py:496: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.2' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[flat_mask, "data_score"] = 3.2
c:\Users\irene.garcia\Documents\DS_analysis_tests\review\notebooks\data_scoring_fns.py:104: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  pd.date_range(x.index.min(), x.index.max(), freq="2T"),
c:\Users\irene.garcia\Documents\DS_analysis_tests\review\notebooks\qa_functions.py:87: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  data["Timestamp_rounded"] = data.index.round(f"{n_mins}T")
c:\Users\irene.garcia\Documents\DS_analysis_tests\review\notebooks\qa_functions.py:87: FutureWarning: 'T' is

,Property_ID,Whole_start,Whole_end,Whole_duration_days,Whole_%_complete_Circulation_Pump_Energy_Consumed,Whole_%_complete_Heat_Pump_Energy_Output,Whole_%_complete_Immersion_Heater_Energy_Consumed,Whole_%_complete_Whole_System_Energy_Consumed,Cleaned_start,Cleaned_end,Cleaned_duration_days,Cleaned_%_complete_Circulation_Pump_Energy_Consumed,Cleaned_%_complete_Heat_Pump_Energy_Output,Cleaned_%_complete_Immersion_Heater_Energy_Consumed,Cleaned_%_complete_Whole_System_Energy_Consumed,Whole_%_complete_Boiler_Energy_Output,Cleaned_%_complete_Boiler_Energy_Output,Whole_%_complete_Back-up_Heater_Energy_Consumed,Cleaned_%_complete_Back-up_Heater_Energy_Consumed,Coldest_day_start,Coldest_day_end,Coldest_day_duration_days,Coldest_day_%_complete_Circulation_Pump_Energy_Consumed,Coldest_day_%_complete_Heat_Pump_Energy_Output,Coldest_day_%_complete_Immersion_Heater_Energy_Consumed,Coldest_day_%_complete_Whole_System_Energy_Consumed,Coldest_day_External_Air_Temperature: 25%,Coldest_day_External_Air_Temperature: 50%,Coldest_day_External_Air_Temperature: 75%,Coldest_day_External_Air_Temperature: count,Coldest_day_External_Air_Temperature: max,Coldest_day_External_Air_Temperature: mean,Coldest_day_External_Air_Temperature: min,Coldest_day_External_Air_Temperature: std,Coldest_day_Internal_Air_Temperature: 25%,Coldest_day_Internal_Air_Temperature: 50%,Coldest_day_Internal_Air_Temperature: 75%,Coldest_day_Internal_Air_Temperature: count,Coldest_day_Internal_Air_Temperature: max,Coldest_day_Internal_Air_Temperature: mean,Coldest_day_Internal_Air_Temperature: min,Coldest_day_Internal_Air_Temperature: std,Coldest_day_Circulation_Pump_Energy_Consumed,Coldest_day_Heat_Pump_Energy_Output,Coldest_day_Immersion_Heater_Energy_Consumed,Coldest_day_Whole_System_Energy_Consumed,Coldest_day_Back-up_Heater_Energy_Consumed,Coldest_day_Boiler_Energy_Output,Coldest_day_spfh2,Coldest_day_spfh3,Coldest_day_spfh4,Coldest_HH_start,Coldest_HH_end,Coldest_HH_duration_days,Coldest_HH_%_complete_Circulation_Pump_Energy_Consumed,Coldest_HH_%_complete_Heat_Pump_Energy_Output,Coldest_HH_%_complete_Immersion_Heater_Energy_Consumed,Coldest_HH_%_complete_Whole_System_Energy_Consumed,Coldest_HH_External_Air_Temperature: 25%,Coldest_HH_External_Air_Temperature: 50%,Coldest_HH_External_Air_Temperature: 75%,Coldest_HH_External_Air_Temperature: count,Coldest_HH_External_Air_Temperature: max,Coldest_HH_External_Air_Temperature: mean,Coldest_HH_External_Air_Temperature: min,Coldest_HH_External_Air_Temperature: std,Coldest_HH_Internal_Air_Temperature: 25%,Coldest_HH_Internal_Air_Temperature: 50%,Coldest_HH_Internal_Air_Temperature: 75%,Coldest_HH_Internal_Air_Temperature: count,Coldest_HH_Internal_Air_Temperature: max,Coldest_HH_Internal_Air_Temperature: mean,Coldest_HH_Internal_Air_Temperature: min,Coldest_HH_Internal_Air_Temperature: std,Coldest_HH_Circulation_Pump_Energy_Consumed,Coldest_HH_Heat_Pump_Energy_Output,Coldest_HH_Immersion_Heater_Energy_Consumed,Coldest_HH_Whole_System_Energy_Consumed,Coldest_HH_Back-up_Heater_Energy_Consumed,Coldest_HH_Boiler_Energy_Output,Coldest_HH_spfh2,Coldest_HH_spfh3,Coldest_HH_spfh4,window_method,acceptable windows: spfh2: count,acceptable windows: spfh2: mean,acceptable windows: spfh2: std,acceptable windows: spfh2: min,acceptable windows: spfh2: 25%,acceptable windows: spfh2: 50%,acceptable windows: spfh2: 75%,acceptable windows: spfh2: max,acceptable windows: spfh3: count,acceptable windows: spfh3: mean,acceptable windows: spfh3: std,acceptable windows: spfh3: min,acceptable windows: spfh3: 25%,acceptable windows: spfh3: 50%,acceptable windows: spfh3: 75%,acceptable windows: spfh3: max,acceptable windows: spfh4: count,acceptable windows: spfh4: mean,acceptable windows: spfh4: std,acceptable windows: spfh4: min,acceptable windows: spfh4: 25%,acceptable windows: spfh4: 50%,acceptable windows: spfh4: 75%,acceptable windows: spfh4: max,window_start,window_end,window_max_gap_score,window_max_data_score,window_max_score,window_mean_gap_score,window

In [7]:
# If running the whole set of homes, we want to over-write the old file
home_summary.to_csv(os.path.join(location_out, "home_summary.csv"), index=False)

# save out all the windows data to file
all_windows.to_csv(os.path.join(location_out, "all_windows.csv"), index=False)

In [8]:
# We save a redacted version of the home summary file to allow a quick comparison to the output of previous code versions
home_summary_partial_2 = home_summary[
    [
        "Property_ID",
        "Whole_start",
        "Whole_end",
        "Whole_duration_days",
        "Whole_%_complete_Circulation_Pump_Energy_Consumed",
        "Whole_%_complete_Whole_System_Energy_Consumed",
        "Whole_%_complete_Heat_Pump_Energy_Output",
        "Whole_%_complete_Immersion_Heater_Energy_Consumed",
        "Cleaned_start",
        "Cleaned_end",
        "Cleaned_duration_days",
        "Cleaned_%_complete_Circulation_Pump_Energy_Consumed",
        "Cleaned_%_complete_Whole_System_Energy_Consumed",
        "Cleaned_%_complete_Heat_Pump_Energy_Output",
        "Cleaned_%_complete_Immersion_Heater_Energy_Consumed",
        "acceptable windows: spfh2: count",
        "acceptable windows: spfh2: mean",
        "acceptable windows: spfh2: std",
        "acceptable windows: spfh2: min",
        "acceptable windows: spfh2: 25%",
        "acceptable windows: spfh2: 50%",
        "acceptable windows: spfh2: 75%",
        "acceptable windows: spfh2: max",
        "acceptable windows: spfh3: count",
        "acceptable windows: spfh3: mean",
        "acceptable windows: spfh3: std",
        "acceptable windows: spfh3: min",
        "acceptable windows: spfh3: 25%",
        "acceptable windows: spfh3: 50%",
        "acceptable windows: spfh3: 75%",
        "acceptable windows: spfh3: max",
        "acceptable windows: spfh4: count",
        "acceptable windows: spfh4: mean",
        "acceptable windows: spfh4: std",
        "acceptable windows: spfh4: min",
        "acceptable windows: spfh4: 25%",
        "acceptable windows: spfh4: 50%",
        "acceptable windows: spfh4: 75%",
        "acceptable windows: spfh4: max",
        "window_start",
        "window_end",
        "window_max_gap_score",
        "window_max_data_score",
        "window_max_score",
        "window_mean_gap_score",
        "window_mean_data_score",
        "window_mean_score",
        "window_Circulation_Pump_Energy_Consumed",
        "window_Whole_System_Energy_Consumed",
        "window_Heat_Pump_Energy_Output",
        "window_Immersion_Heater_Energy_Consumed",
        "window_Back-up_Heater_Energy_Consumed",
        "window_Boiler_Energy_Output",
        "spfh2",
        "spfh3",
        "spfh4",
        "window_duration_days",
        "window_%_complete_Circulation_Pump_Energy_Consumed",
        "window_%_complete_Whole_System_Energy_Consumed",
        "window_%_complete_Heat_Pump_Energy_Output",
        "window_%_complete_Immersion_Heater_Energy_Consumed",
        "window_method",
        "Whole_%_complete_Boiler_Energy_Output",
        "Cleaned_%_complete_Boiler_Energy_Output",
        "window_%_complete_Boiler_Energy_Output",
        "Whole_%_complete_Back-up_Heater_Energy_Consumed",
        "Cleaned_%_complete_Back-up_Heater_Energy_Consumed",
        "window_%_complete_Back-up_Heater_Energy_Consumed",
    ]
]
home_summary_partial_2.to_csv(os.path.join(location_out, "home_summary_partial_2.csv"), index=False)